In [1]:
# this script adds Sales Month and Sales Quarter in unified format to all data until and including 2024 Q4


import pandas as pd
import os

input1 = "../Earth/Combined statements/old/Earth_2022Q4_2024Q4_1_raw_combined_fixed.csv"
outputfilename = "Earth_2022Q4_2024Q4_1_raw_combined_Salesmonthandquarter.csv"

dtype_1 = {
   'colum_1' : 'str',
    'colum_2' : 'str',
    'colum_3' : 'str',
    'colum_4' : 'str',
    'colum_5' : 'str',
    'colum_6' : 'str',
    'colum_7' : 'str',
    'colum_8' : 'str',
    'colum_9' : 'float64',
    'colum_10' : 'float64',
    'colum_11' : 'str',
    'colum_12' : 'str',
    'colum_13' : 'str',
    'colum_14' : 'str',
    'colum_15' : 'str',
    'colum_16' : 'str',
    'colum_17' : 'str',
    'colum_18' : 'str',
    'colum_19' : 'str',
    'colum_20' : 'str',
    'colum_21' : 'float64',
    'colum_22' : 'str',
    'colum_23' : 'str',
    'colum_24' : 'float64',
    'colum_25' : 'float64',
    'colum_26' : 'float64',
    'colum_27' : 'float64',
    'colum_28' : 'str',
    'colum_29' : 'str',
    'colum_30' : 'str',
    'colum_31' : 'str',
    'colum_32' : 'float64',
    'colum_33' : 'float64',
    'colum_34' : 'float64',
    'colum_35' : 'float64',
    'colum_36' : 'float64',
    'colum_37' : 'float64',
    'colum_38' : 'str',
    'colum_39' : 'str',
    'colum_40' : 'str',
    'colum_41' : 'str',
    'colum_42' : 'str',
    'colum_43' : 'str',
    'colum_44' : 'str',
    'colum_45' : 'float64',
    'colum_46' : 'float64'
}


df = pd.read_csv(input1,dtype=dtype_1,low_memory=False)
print(f"DataFrame: Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Total fee: {df['Royalties (USD)'].sum()}. Total units: {df['Units'].sum()}")

df = df.drop(columns=['实际分成收入(TWD)','总计','Payable CNY'],errors = 'ignore')
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

#for column in df.columns:        
#        print(column)

all_dates = pd.concat([df["Period"], df["Period start"]])

# Define a function to parse various formats
def parse_date(value):
    value = str(value).strip()
    # fmts = ["%Y/%m/%d", "%Y%m%d", "%Y%m", "%Y %m"]
    fmts = ["%Y/%m/%d", "%Y%m%d", "%Y%m", "%Y %m", "%d/%m/%Y  %H:%M:%S"]

    for fmt in fmts:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue
    # If parsing fails, return NaT
    return pd.NaT

# Apply parsing to each column
df["Period_dt"] = df["Period"].apply(parse_date)
df["Period_start_dt"] = df["Period start"].apply(parse_date)

# You can pick which column to use (e.g., Period start first if available), here as an example:
df["Sales_date"] = df["Period_start_dt"].combine_first(df["Period_dt"])

# Format Sales Month: "YYYY MM"
df["Sales Month"] = df["Sales_date"].dt.strftime("%Y %m")

# Format Sales Quarter: "YYYY QN"
df["Sales Quarter"] = (
    df["Sales_date"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
)

df['Sales Quarter']=df['Sales Quarter'].str.replace('NaT','9999')
df.fillna({'Sales Month': '9999'}, inplace=True)

df.rename(columns={"Statement": "Statement Quarter"}, inplace=True)
df = df.drop(columns=['Period_dt','Period_start_dt','Sales_date'],errors = 'ignore')
df = df.sort_index(axis=1)
print(f"Total fee: {df['Royalties (USD)'].sum()}. Total units: {df['Units'].sum()}")
for column in df.columns:        
        print(column)



DataFrame: Rows: 991290, Columns: 56
Total fee: 378369.37166618125. Total units: 239471829.0
Total fee: 378369.37166618125. Total units: 239471829.0
Album
Artist
Composer Name
Content
Copyright holder unique code
Country
Currency
Device
FX Rate
Gross Amount
ISRC
Internal costs
Issuance
Noise Content
Paid/not paid
Period
Period end
Period start
Platform
Price
Product
Product Type Identifier
Royalties
Royalties (CNY)
Royalties (USD)
Royalty Rate
Sales Month
Sales Quarter
Sales or Return
Settlement type
Share
Share Lyricist
Share MABB (CNY)
Share MABB (USD)
Share Master Owner
Share composer
Song
Song ID
Song Length
Statement Quarter
Streaming Subscription Category
Streaming Subscription Type
Streaming category
Streaming type
UPC
Units
Withholding Tax


In [2]:

df.to_csv(outputfilename, index=False)